In [ ]:
import pandas as pd
import numpy  as np

loans = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])
members = pd.read_csv('data/library_members.csv')
merged = pd.read_csv('data/merged.csv', parse_dates=['loan_date', 'return_date'])
merged.shape

## Lab 4 - Part 2
### Ex 1

In [ ]:
no_member_data = merged[merged['branch'] == 'No member record']

no_member_data.agg(n=('loan_id','count'),fee=('late_fee','sum'))

In [ ]:
no_member_data.groupby('genre').agg(num=('loan_id','count'))

#### 31 loans, fee: 347.5. genres: Children 10, Sci-fi 5, Crime 4, Fantasy 4, Romance 4, History 4
#### Genre slightly favoured to Children but overall no clear pattern.
#### Ask whoever enters the member data how 6 members were not entered correctly.
### Ex 2

In [ ]:
outer_merge = loans.merge(members, on='member_id', how='outer', indicator=True)
right_only = outer_merge['_merge'] == 'right_only'

print(outer_merge['_merge'].value_counts())
outer_merge[right_only]

In [ ]:
members[~members['member_id'].isin(loans['member_id'])]

#### indicator data from merge: both: 585, left_only: 31, right_only: 1
#### Prefer the isin approach. The merge will only contain null values and is more work.
### Ex 3

In [ ]:
year_delta = merged['loan_date'].dt.year - merged['birth_year']
merged['year_delta'] = year_delta #added col year_delta to more easily work with this data

print(merged.loc[year_delta.isna(),['birth_year','year_delta']].shape)
year_delta.describe()

In [ ]:
merged.groupby('genre')['year_delta'].agg(['count','mean']).sort_values('mean').round(1)

#### 143 loans have no birth_year, loan_date or no member and therefore no delta. Mean age: 38.77y.
#### Genres: History 35.2 (45 loans), Sci-fi 37.8 (66), Romance 38.1 (58), Crime 38.4 (120), Fantasy 39.8 (67), Children 41.1 (101)
#### The average age seem inverted to what could be expected. Children highest at 41.1y and History lowest at 35.2y.
### Ex 4

#### Missing branch: We decided to fill these with a placeholder value describing the two cases of missing branch. 'Unknown' for not registered branch and 'No member record' when loans had no recorded member. Fill.
#### Missing birth_year: We decided to leave these NaN as filling with median values would lessen the data quality. By leaving these NaN they are excluded for any aggragate function calculations. Leave.
#### Faulty birth_year (1900,2030): These were identified as incorrect entries and exchanged for NaN values as the correct data is missing. Fill.
#### Duplicates: After identifying that they are true duplicates with identical data and adjacent entries. Remove.